In [1]:
import pandas as pd
import numpy as np
def pick_hours(night, n_hours=4):

    # Sort by signal first, then by time, so a tie is won by the earlier hour
    ranked = night.sort_values(["signal", "hour_start_nem"])
    # Keep the n_hours best rows
    chosen = ranked.head(n_hours)
    # Put them back in time order so the result is easy to read
    return chosen.sort_values("hour_start_nem")

def pick_fixed(night, n_hours=4):
    """Baseline: plug in at 18:00 and charge straight away (no signal)."""
    # The first n_hours of the night in time order = 18:00, 19:00, 20:00, 21:00
    return night.sort_values("hour_start_nem").head(n_hours)

In [2]:
night = pd.DataFrame({
    "hour_start_nem":    pd.date_range("2025-07-01 18:00", periods=13, freq="h"),
    "hour_start_sydney": pd.date_range("2025-07-01 18:00", periods=13, freq="h"),
    #           18    19    20    21    22    23    00    01    02    03    04    05    06
    "signal": [0.50, 0.60, 0.70, 0.80, 0.30, 0.20, 0.10, 0.40, 0.40, 0.90, 0.95, 0.85, 0.75],
})

# By hand: the lowest are 00:00 (0.10), 23:00 (0.20), 22:00 (0.30).
# The 4th place is a tie between 01:00 and 02:00 (both 0.40), so the earliest, 01:00, wins.
expected = [22, 23, 0, 1]

picked = pick_hours(night)
assert list(picked["hour_start_sydney"].dt.hour) == expected, "scheduler picked the wrong hours"
assert len(picked) == 4

fixed = pick_fixed(night)
assert list(fixed["hour_start_sydney"].dt.hour) == [18, 19, 20, 21], "fixed baseline is wrong"

print("All tests passed")

All tests passed


In [3]:
def load_hourly(path):
    """Read a shared hourly table (MEF, AEI or price). Times stay as plain local clock times."""
    return pd.read_csv(path, parse_dates=["hour_start_nem", "hour_start_sydney"])


def expected_hours(night_of):
    """How many real hours the night 18:00 on D to 06:00 on D+1 has (13, or 12 / 14 at daylight saving)."""
    start = (night_of + pd.Timedelta(hours=18)).tz_localize("Australia/Sydney")
    end = (night_of + pd.Timedelta(days=1, hours=6)).tz_localize("Australia/Sydney")
    return int((end - start) / pd.Timedelta(hours=1)) + 1


def slice_nights(df):
    """Keep only hours inside the 18:00-06:00 window, label each with night_of, keep complete nights."""
    hour = df["hour_start_sydney"].dt.hour
    df = df[(hour >= 18) | (hour <= 6)].copy()
    # Subtract 18 h: 18:00 on D -> 00:00 on D, 06:00 on D+1 -> 12:00 on D. Then take the date.
    df["night_of"] = (df["hour_start_sydney"] - pd.Timedelta(hours=18)).dt.normalize()
    counts = df.groupby("night_of")["hour_start_nem"].transform("count")
    expected = df["night_of"].map(expected_hours)
    return df[counts == expected].sort_values("hour_start_nem")

In [4]:
# Fake year in NEM time, then convert to Sydney time the way A5 says
nem = pd.date_range("2025-07-01 00:00", "2026-06-30 23:00", freq="h")
fake = pd.DataFrame({"hour_start_nem": nem})
fake["hour_start_sydney"] = (nem.tz_localize("Australia/Brisbane")
                                .tz_convert("Australia/Sydney").tz_localize(None))
fake["signal"] = np.random.rand(len(fake))

nights = slice_nights(fake)
sizes = nights.groupby("night_of").size()

assert len(fake) == 8760
assert len(sizes) == 364, len(sizes)                      # 1 Jul 2025 to 29 Jun 2026
assert sizes.loc["2025-10-04"] == 12 and sizes.loc["2026-04-04"] == 14
assert (sizes.drop([pd.Timestamp("2025-10-04"), pd.Timestamp("2026-04-04")]) == 13).all()

# Run the Step 1 picker on every night
for d, night in nights.groupby("night_of"):
    assert len(pick_hours(night)) == 4
    assert list(pick_fixed(night)["hour_start_sydney"].dt.hour) == [18, 19, 20, 21]
print("Step 2 tests passed")

Step 2 tests passed


In [5]:
price = slice_nights(load_hourly("price_hourly_nsw1_v1.csv"))
aei = slice_nights(load_hourly("aei_hourly_nsw1_v1.csv"))

print("price nights:", price["night_of"].nunique())   # expect 364
print("aei nights:  ", aei["night_of"].nunique())     # expect 364

# Hours per night: expect 13 almost everywhere, 12 on 2025-10-04, 14 on 2026-04-04
print(price.groupby("night_of").size().value_counts())

price nights: 364
aei nights:   364
13    362
12      1
14      1
Name: count, dtype: int64


In [6]:
KWH_PER_HOUR = 9.2 / 4   # 2.3 kWh in each of the 4 charging hours

# 1. Join price, AEI and the three MEF versions on hour_start_nem
hours = price[["hour_start_nem", "hour_start_sydney", "night_of", "value"]].rename(columns={"value": "price"})
hours = hours.merge(aei[["hour_start_nem", "value"]].rename(columns={"value": "aei"}), on="hour_start_nem")
for v in ["B", "C", "Cfwd"]:
    m = load_hourly(f"mef_hourly_{v}_rules-v1.csv")[["hour_start_nem", "value"]]
    hours = hours.merge(m.rename(columns={"value": f"mef_{v}"}), on="hour_start_nem")

# 2. Missing-night rule: skip any night with an empty value
check = ["price", "aei", "mef_B", "mef_C", "mef_Cfwd"]
bad = hours.loc[hours[check].isna().any(axis=1), "night_of"].unique()
hours = hours[~hours["night_of"].isin(bad)]
print("nights skipped:", len(bad), "| nights used:", hours["night_of"].nunique())


def run_strategy(hours, name, signal_col=None):
    """Pick 4 hours per night, then score with the ACTUAL MEF (B, C, C-fwd), price and AEI."""
    rows = []
    for d, night in hours.groupby("night_of"):
        if signal_col is None:
            chosen = pick_fixed(night)
        else:
            chosen = pick_hours(night.assign(signal=night[signal_col]))
        k = KWH_PER_HOUR
        rows.append({
            "night_of": d,
            "strategy": name,
            "hours_chosen": ";".join(str(h) for h in chosen["hour_start_sydney"].dt.hour),
            "energy_kwh": k * len(chosen),
            "cost_aud": (k * chosen["price"]).sum() / 1000,
            "emissions_kg_B": (k * chosen["mef_B"]).sum(),
            "emissions_kg_C": (k * chosen["mef_C"]).sum(),
            "emissions_kg_Cfwd": (k * chosen["mef_Cfwd"]).sum(),
            "aei_accounted_kg": (k * chosen["aei"]).sum(),
            "_set": frozenset(chosen["hour_start_nem"]),   # for the overlap check only
        })
    return pd.DataFrame(rows)


# 3. Run the four strategies (mef_oracle picks by version C)
res = {name: run_strategy(hours, name, col) for name, col in
       [("mef_oracle", "mef_C"), ("aei", "aei"), ("price", "price"), ("fixed", None)]}
results = pd.concat(res.values())
assert np.allclose(results["energy_kwh"], 9.2)

# 4. Totals per strategy over all nights
cols = ["cost_aud", "emissions_kg_B", "emissions_kg_C", "emissions_kg_Cfwd", "aei_accounted_kg"]
print(results.groupby("strategy")[cols].sum().round(1).to_string())

# 5. Overlap: share of nights each signal picked exactly the same 4 hours as mef_oracle
for name in ["aei", "price", "fixed"]:
    same = (res[name]["_set"].values == res["mef_oracle"]["_set"].values).mean()
    print(f"{name} picks the oracle's 4 hours on {same * 100:.1f}% of nights")

# 6. Save the results (drop the _set helper column, it was only for the overlap check)
results.drop(columns="_set").to_csv("schedule_results_v1.csv", index=False)

nights skipped: 0 | nights used: 364


            cost_aud  emissions_kg_B  emissions_kg_C  emissions_kg_Cfwd  aei_accounted_kg
strategy                                                                                 
aei            388.1          1904.8          1952.4             2151.4            2135.8
fixed          426.4          1646.3          1696.3             2072.7            2198.1
mef_oracle     368.9          1448.8          1345.0             1722.5            2262.0
price          250.2          2321.7          2274.1             2306.7            2267.2
aei picks the oracle's 4 hours on 3.0% of nights
price picks the oracle's 4 hours on 0.0% of nights
fixed picks the oracle's 4 hours on 11.5% of nights


In [7]:
FORECAST_FILE = "mef_forecast_C_rules-v1_ridge-v1.csv"

fc = pd.read_csv(FORECAST_FILE, parse_dates=["night_of", "hour_start_sydney", "hour_start_nem"])
fc = fc[fc["split"] == "test"]                       # test nights only; validation rows are not scheduled
print("forecast test nights:", fc["night_of"].nunique(), "| rows:", len(fc))    # expect 53 | 689

# Join the forecast onto the Round 1 table (actuals like mef_C, aei, price are already in `hours`)
test = hours.merge(
    fc[["hour_start_nem", "night_of", "mef_pred"]].rename(columns={"night_of": "fc_night"}),
    on="hour_start_nem", 
    how="inner"
)

assert (test["night_of"] == test["fc_night"]).all(), "night_of differs between my slicer and the forecast file"
assert test["mef_pred"].notna().all()
assert test["night_of"].nunique() == 53

# Same scheduler: pick the 4 lowest FORECAST hours, then score with the ACTUAL values
res2 = {name: run_strategy(test, name, col) for name, col in
        [("mef_forecast", "mef_pred"), ("mef_oracle", "mef_C"), ("aei", "aei"), ("price", "price"), ("fixed", None)]}
r2 = pd.concat(res2.values())
assert np.allclose(r2["energy_kwh"], 9.2)

cols = ["cost_aud", "emissions_kg_B", "emissions_kg_C", "emissions_kg_Cfwd", "aei_accounted_kg"]
print(r2.groupby("strategy")[cols].sum().round(1).to_string())

# Nightly regret = emissions(mef_forecast) - emissions(mef_oracle), under each MEF version
regret = pd.DataFrame({
    v: res2["mef_forecast"].set_index("night_of")[f"emissions_kg_{v}"]
       - res2["mef_oracle"].set_index("night_of")[f"emissions_kg_{v}"]
    for v in ["B", "C", "Cfwd"]})
print(regret.describe().round(3).to_string())
print("total regret kg:", regret.sum().round(1).to_dict())
same_share = (res2["mef_forecast"]["_set"].values == res2["mef_oracle"]["_set"].values).mean()
print("share of nights forecast picked the oracle's 4 hours:", round(same_share * 100, 1), "%")





forecast test nights: 53 | rows: 689


              cost_aud  emissions_kg_B  emissions_kg_C  emissions_kg_Cfwd  aei_accounted_kg
strategy                                                                                   
aei               41.7           321.8           322.3              333.6             320.5
fixed             47.7           280.3           301.9              339.9             333.5
mef_forecast      46.9           283.0           301.4              337.8             333.3
mef_oracle        43.6           250.2           245.1              273.8             332.5
price             37.5           338.5           323.7              329.8             328.5
            B       C    Cfwd
count  53.000  53.000  53.000
mean    0.619   1.061   1.206
std     0.890   0.680   0.717
min    -1.158   0.000  -0.165
25%     0.000   0.547   0.794
50%     0.627   0.984   1.096
75%     1.121   1.443   1.433
max     2.634   3.090   3.586
total regret kg: {'B': 32.8, 'C': 56.2, 'Cfwd': 63.9}
share of nights forecast picked 

In [8]:
# Add the mef_forecast rows (test nights only) to the results file.
# Rebuild the file from the Round 1 table every time, so running this cell twice never adds the rows twice.
forecast_rows = res2["mef_forecast"].drop(columns="_set")
all_rows = pd.concat([results.drop(columns="_set"), forecast_rows])

# Expect 364 nights for each Round 1 strategy and 53 test nights for mef_forecast
counts = all_rows.groupby("strategy").size()
print(counts.to_string())
assert counts.drop("mef_forecast").eq(364).all() and counts["mef_forecast"] == 53

all_rows.to_csv("schedule_results_v1.csv", index=False)
print("saved schedule_results_v1.csv with", len(all_rows), "rows")

strategy
aei             364
fixed           364
mef_forecast     53
mef_oracle      364
price           364
saved schedule_results_v1.csv with 1509 rows


In [9]:
# Test-night files for the team: Ridge forecast vs fixed in one file, oracle kept in its own file.
# All rows come from res2, so every strategy is on the same 53 test nights.
ridge_vs_fixed = pd.concat([res2["mef_forecast"], res2["fixed"]]).drop(columns="_set")
oracle_test = res2["mef_oracle"].drop(columns="_set")

# Checks: 53 nights per strategy, the same nights in both files, 9.2 kWh delivered every night
assert ridge_vs_fixed.groupby("strategy")["night_of"].nunique().eq(53).all()
assert len(ridge_vs_fixed) == 2 * 53 and len(oracle_test) == 53
assert set(oracle_test["night_of"]) == set(ridge_vs_fixed["night_of"])
assert np.allclose(ridge_vs_fixed["energy_kwh"], 9.2) and np.allclose(oracle_test["energy_kwh"], 9.2)

ridge_vs_fixed.to_csv("schedule_test_ridge_vs_fixed_v1.csv", index=False)
oracle_test.to_csv("schedule_test_oracle_v1.csv", index=False)
print("saved", len(ridge_vs_fixed), "Ridge/fixed rows and", len(oracle_test), "oracle rows; 9.2 kWh on every night")

saved 106 Ridge/fixed rows and 53 oracle rows; 9.2 kWh on every night
